# Predicting viability from image-based profiles using Elastic Net regression

## Imports, pathing, and constants

In [ ]:
import logging
import pathlib
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import umap
from notebook_init_utils import bandicoot_check, init_notebook
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path("/home/lippincm/mnt/bandicoot").resolve(), root_dir
)

/home/lippincm/Documents/NF1_organoid_profile_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# set up logging
year_month_day_hour_minute_log_name = (
    f"{pd.Timestamp.now().strftime('%Y-%m-%d_%H-%M')}_viability_prediction_training.log"
)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(f"../logs/{year_month_day_hour_minute_log_name}.log"),
        logging.StreamHandler(sys.stdout),
    ],
)

In [2]:
"""
Elastic Net viability model: 
training + evaluation under three split strategies

  1. "lopo"          - Leave-One-Patient-Out cross-validation
  2. "loto"          - Leave-One-Treatment-Out cross-validation
  3. "random_split"  - a single random 70/30 train/test split (no grouping)

All three reuse the same cleaning / training / metrics / artifact-saving
logic. For "lopo" and "loto", the model is refit once per fold (holding
out all rows for one patient / one treatment as the test set), metrics
are computed per fold, and results are aggregated across folds. For
"random_split", the model is fit once on a random 70% of rows and
evaluated on the held-out 30%, with the same metrics/artifact format so
it can be compared side-by-side with the grouped results.

Predicted viability is always bounded to [0, 100] before it's reported
or scored, since viability is a percentage and the raw ElasticNet
output is unconstrained and can fall outside that range.
"""

# ---------------------------------------------------------------------
# CONFIG - update these to match your data
# ---------------------------------------------------------------------
PATIENT_COL = "Metadata_Biology_PatientTumor"  # column identifying each patient
TREATMENT_COL = "Metadata_Experiment_FullTreatment"  # column identifying each treatment
SPLIT_COL = "Metadata_data_split"  # used only for split_method="predefined"

RANDOM_SPLIT_TEST_SIZE = 0.3  # fraction held out for the random_split test set
RANDOM_SPLIT_SEED = 0  # fixed seed so the random split is reproducible

VIABILITY_MIN = 0.0
VIABILITY_MAX = 100.0

MODEL_OUTPUT = pathlib.Path("../trained_models")
RESULTS_OUTPUT = pathlib.Path("../model_results")
MODEL_OUTPUT.mkdir(exist_ok=True)
RESULTS_OUTPUT.mkdir(exist_ok=True)


def clip_predictions(preds):
    """Bound predicted viability to a physically meaningful [0, 100] range."""
    return np.clip(preds, VIABILITY_MIN, VIABILITY_MAX)


# ---------------------------------------------------------------------
# Data cleaning (from your original script)
# ---------------------------------------------------------------------
def clean_features(df, feature_cols):
    """
    Reports inf/NaN diagnostics for feature_cols, then replaces inf with NaN,
    drops rows with NaN in those columns, and clips extreme values.
    Returns the cleaned dataframe (fewer rows if any contained inf/NaN).
    """
    X = df[feature_cols]

    print("Any inf in X:", np.isinf(X.values).any())
    print("Any NaN in X:", np.isnan(X.values).any())
    print("Max abs value in X:", np.nanmax(np.abs(X.values)))

    # Index into X's own columns, not df.columns — the boolean mask
    # has one entry per feature column, not per column in df.
    inf_mask = np.isinf(X.values).any(axis=0)
    inf_cols = X.columns[inf_mask]
    print("Columns with inf values:", list(inf_cols))

    # Option A: replace inf with NaN, then drop rows with NaN
    X = X.replace([np.inf, -np.inf], np.nan)
    valid_rows = X.dropna().index
    df = df.loc[valid_rows].copy()
    X = X.loc[valid_rows]

    # Option B: clip extreme values instead of dropping
    X = X.clip(lower=-1e10, upper=1e10)
    df[feature_cols] = X

    return df.reset_index(drop=True)


# ---------------------------------------------------------------------
# Model training (from your original script)
# ---------------------------------------------------------------------
def train_elastic_net(X_train: pd.DataFrame, Y_train: pd.Series) -> "Pipeline":
    elastic_net_model = make_pipeline(
        StandardScaler(),
        ElasticNetCV(
            l1_ratio=[0.1, 0.2, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
            alphas=100,  # int -> auto-generates 100 alphas along the path
            cv=5,
            random_state=0,
            max_iter=50000,
            tol=1e-4,
            n_jobs=-1,
        ),
    )
    elastic_net_model.fit(X_train, Y_train.values.ravel())
    return elastic_net_model


# ---------------------------------------------------------------------
# Metrics (from your original script)
# ---------------------------------------------------------------------
def compute_metrics(model, X, Y) -> dict:
    y_pred = clip_predictions(model.predict(X))
    y_true = Y.values.ravel()
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rmse = np.sqrt(mse)
    return {"R2": r2, "MSE": mse, "MAE": mae, "MAPE": mape, "RMSE": rmse}


# ---------------------------------------------------------------------
# Strategy 1 & 2: grouped CV (LOPO / LOTO), reusing the same core logic
# ---------------------------------------------------------------------
def run_group_cv(
    viabilities_df: pd.DataFrame,
    feature_cols: list,
    viability_col: str,
    group_col: str,
    split_name: str,
    **kwargs,
) -> pd.DataFrame:
    """
    Runs Leave-One-Group-Out CV (group_col = patient or treatment column),
    refitting the elastic net pipeline each fold, and saving per-fold and
    aggregated artifacts under OUTPUT_DIR.
    """
    list_of_metadatas = []

    if "shuffle_status" in kwargs:
        shuffle_status = kwargs["shuffle_status"]
        list_of_metadatas.append(shuffle_status)
    else:
        shuffle_status = "not_shuffled"
    if "profile_type" in kwargs:
        profile_type = kwargs["profile_type"]
        list_of_metadatas.append(profile_type)
    tag = "__".join(list_of_metadatas) if list_of_metadatas else split_name

    # Clean BEFORE computing groups, so groups/train_idx/test_idx all line
    # up with the same (possibly row-dropped) dataframe.
    viabilities_df = clean_features(viabilities_df, feature_cols)
    groups = viabilities_df[group_col].values

    logo = LeaveOneGroupOut()

    all_metrics = []
    all_predictions = []
    all_importances = []

    n_splits = logo.get_n_splits(groups=groups)
    print(
        f"\n=== {split_name} grouped CV on '{group_col}' ({n_splits} folds) for {shuffle_status} ==="
    )

    for fold_idx, (train_idx, test_idx) in enumerate(
        logo.split(viabilities_df, groups=groups)
    ):
        held_out = np.unique(groups[test_idx])[0]

        train_df = viabilities_df.iloc[train_idx]
        test_df = viabilities_df.iloc[test_idx]

        X_train, X_test = train_df[feature_cols], test_df[feature_cols]
        Y_train, Y_test = train_df[viability_col], test_df[viability_col]

        # call custom training function to fit the model
        model = train_elastic_net(X_train, Y_train)

        for eval_split, (X_eval, Y_eval) in {
            "train": (X_train, Y_train),
            "test": (X_test, Y_test),
        }.items():
            m = compute_metrics(model, X_eval, Y_eval)
            m.update(
                {
                    "Metadata_fold": fold_idx,
                    "Metadata_held_out_group": held_out,
                    "Metadata_eval_split": eval_split,
                    "Metadata_n_samples": len(X_eval),
                }
            )
            all_metrics.append(m)

            print(
                f"  Fold {fold_idx} (held out={held_out}, n={len(X_eval)}, {eval_split}): "
                f"R2={m['R2']:.4f}, RMSE={m['RMSE']:.4f}"
            )

        # Everything below runs once per FOLD (not once per eval_split) -
        # the model/predictions/coefficients don't change between the
        # train-eval and test-eval passes above.

        # Held-out predictions (bounded to [0, 100] since viability is a percentage)
        fold_preds = test_df.copy()
        fold_preds["Actual_Viability"] = Y_test.values
        fold_preds["Predicted_Viability"] = clip_predictions(model.predict(X_test))
        fold_preds["Metadata_fold"] = fold_idx
        fold_preds["Metadata_held_out_group"] = held_out
        all_predictions.append(fold_preds)

        # Feature importances for this fold
        fold_importance = pd.DataFrame(
            {
                "feature": feature_cols,
                "importance": model.named_steps["elasticnetcv"].coef_,
                "Metadata_fold": fold_idx,
                "Metadata_held_out_group": held_out,
                "Metadata_shuffle_status": shuffle_status,
            }
        )
        all_importances.append(fold_importance)

        # Save the fold model (comment out if this produces too many files)
        joblib.dump(
            model,
            MODEL_OUTPUT
            / f"{split_name}_model_fold{fold_idx}_{held_out}__{tag}.joblib",
        )

    metrics_df = pd.DataFrame(all_metrics)[
        [
            "Metadata_fold",
            "Metadata_held_out_group",
            "Metadata_eval_split",
            "Metadata_n_samples",
            "R2",
            "MSE",
            "MAE",
            "MAPE",
            "RMSE",
        ]
    ]
    metrics_df.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_model_performance__{tag}.parquet", index=False
    )

    predictions_df = pd.concat(all_predictions, ignore_index=True)
    predictions_df.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_predicted_viabilities__{tag}.parquet",
        index=False,
    )

    importances_df = pd.concat(all_importances, ignore_index=True)
    importances_df.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_feature_importances__{tag}.parquet", index=False
    )

    # Aggregate summary (mean/std across folds, test set only)
    test_metrics = metrics_df[metrics_df["Metadata_eval_split"] == "test"]
    summary = test_metrics[["R2", "MSE", "MAE", "MAPE", "RMSE"]].agg(["mean", "std"])
    print(f"\n--- {split_name} summary (across {n_splits} folds, test set) ---")
    print(summary)
    summary.to_parquet(RESULTS_OUTPUT / f"{split_name}_summary_metrics__{tag}.parquet")

    return metrics_df


# ---------------------------------------------------------------------
# Strategy 3: single random 70/30 train/test split (no grouping)
# ---------------------------------------------------------------------
def run_random_split(
    viabilities_df: pd.DataFrame,
    feature_cols: list,
    viability_col: str,
    split_name: str = "random_split",
    test_size: float = RANDOM_SPLIT_TEST_SIZE,
    random_state: int = RANDOM_SPLIT_SEED,
    **kwargs,
) -> pd.DataFrame:
    """
    Trains once on a random (test_size fraction) held-out split rather than
    grouping by patient or treatment. Rows are shuffled and split
    independently of any Metadata_* grouping column, so the same
    patient/treatment can appear in both train and test — this is meant as
    a baseline to compare against the stricter LOPO/LOTO splits, not a
    substitute for them.

    Saves the same artifact shapes (metrics/predictions/importances/model)
    as run_group_cv, using "Metadata_fold" = 0 and a fixed
    "Metadata_held_out_group" label, so results from both strategies can be
    concatenated and compared directly.
    """
    list_of_metadatas = []

    if "shuffle_status" in kwargs:
        shuffle_status = kwargs["shuffle_status"]
        list_of_metadatas.append(shuffle_status)
    else:
        shuffle_status = "not_shuffled"
    if "profile_type" in kwargs:
        profile_type = kwargs["profile_type"]
        list_of_metadatas.append(profile_type)
    tag = "__".join(list_of_metadatas) if list_of_metadatas else split_name

    viabilities_df = clean_features(viabilities_df, feature_cols)

    held_out_label = f"random_{int(round(test_size * 100))}pct_holdout"

    train_df, test_df = train_test_split(
        viabilities_df, test_size=test_size, random_state=random_state
    )

    print(
        f"\n=== {split_name} ({int(round((1 - test_size) * 100))}/{int(round(test_size * 100))} "
        f"train/test, n_train={len(train_df)}, n_test={len(test_df)}) for {shuffle_status} ==="
    )

    X_train, X_test = train_df[feature_cols], test_df[feature_cols]
    Y_train, Y_test = train_df[viability_col], test_df[viability_col]

    model = train_elastic_net(X_train, Y_train)

    all_metrics = []
    for eval_split, (X_eval, Y_eval) in {
        "train": (X_train, Y_train),
        "test": (X_test, Y_test),
    }.items():
        m = compute_metrics(model, X_eval, Y_eval)
        m.update(
            {
                "Metadata_fold": 0,
                "Metadata_held_out_group": held_out_label,
                "Metadata_eval_split": eval_split,
                "Metadata_n_samples": len(X_eval),
            }
        )
        all_metrics.append(m)
        print(
            f"  {eval_split} (n={len(X_eval)}): R2={m['R2']:.4f}, RMSE={m['RMSE']:.4f}"
        )

    # Held-out predictions (bounded to [0, 100] since viability is a percentage)
    fold_preds = test_df.copy()
    fold_preds["Actual_Viability"] = Y_test.values
    fold_preds["Predicted_Viability"] = clip_predictions(model.predict(X_test))
    fold_preds["Metadata_fold"] = 0
    fold_preds["Metadata_held_out_group"] = held_out_label

    fold_importance = pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": model.named_steps["elasticnetcv"].coef_,
            "Metadata_fold": 0,
            "Metadata_held_out_group": held_out_label,
            "Metadata_shuffle_status": shuffle_status,
        }
    )

    joblib.dump(model, MODEL_OUTPUT / f"{split_name}_model__{tag}.joblib")

    metrics_df = pd.DataFrame(all_metrics)[
        [
            "Metadata_fold",
            "Metadata_held_out_group",
            "Metadata_eval_split",
            "Metadata_n_samples",
            "R2",
            "MSE",
            "MAE",
            "MAPE",
            "RMSE",
        ]
    ]
    metrics_df.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_model_performance__{tag}.parquet", index=False
    )
    fold_preds.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_predicted_viabilities__{tag}.parquet",
        index=False,
    )
    fold_importance.to_parquet(
        RESULTS_OUTPUT / f"{split_name}_feature_importances__{tag}.parquet", index=False
    )

    # Single-split summary (std will be NaN with only one test evaluation - that's expected)
    test_metrics = metrics_df[metrics_df["Metadata_eval_split"] == "test"]
    summary = test_metrics[["R2", "MSE", "MAE", "MAPE", "RMSE"]].agg(["mean", "std"])
    print(f"\n--- {split_name} summary (single split, test set) ---")
    print(summary)
    summary.to_parquet(RESULTS_OUTPUT / f"{split_name}_summary_metrics__{tag}.parquet")

    return metrics_df

In [3]:
patient_ids = pd.read_csv(
    pathlib.Path(f"{root_dir}/data/patient_IDs.txt").resolve(strict=True),
    header=None,
    sep="\t",
    names=["patient_id"],
)["patient_id"].to_list()

viabilities_path = pathlib.Path(f"{root_dir}/data/viabilities/").resolve(strict=True)

## Combine the profiles, viabilities, and platemap information

In [4]:
platemap_df_list = []
for patient in patient_ids:
    platemap_file_path = pathlib.Path(
        f"{root_dir}/config/platemaps/{patient}_platemap.csv"
    ).resolve(strict=True)
    tmp_df = pd.read_csv(platemap_file_path, index_col=0)
    tmp_df["patient_id"] = patient
    platemap_df_list.append(tmp_df)
platemap_df = pd.concat(platemap_df_list, axis=0)

In [5]:
viabilities_df_list = []
for patient in patient_ids:
    viabilities_file_path = pathlib.Path(
        f"{root_dir}/data/viabilities/{patient}_Viabilities.csv"
    ).resolve()
    if not viabilities_file_path.exists():
        continue
    viabilities_df = pd.read_csv(viabilities_file_path)
    # change DMSO dose to 1
    viabilities_df.loc[viabilities_df["Drug"] == "DMSO", "Concentration_uM"] = 1
    viabilities_df.loc[viabilities_df["Drug"] == "PD0325901", "Drug"] = "Mirdametinib"
    viabilities_df["patient_id"] = patient

    viabilities_df_list.append(viabilities_df)
viabilities_df = pd.concat(viabilities_df_list, axis=0)

In [6]:
# merge the viabilities with the platemap
platemap_viability_df = pd.merge(
    platemap_df,
    viabilities_df,
    how="left",
    left_on=["Treatment", "Dose", "patient_id"],
    right_on=["Drug", "Concentration_uM", "patient_id"],
).drop(columns=["WellCol", "WellPosition"])
# check if the NANs are in B wells only
nan_rows = platemap_viability_df[platemap_viability_df.isna().any(axis=1)]
# drop nan rows
platemap_viability_df = platemap_viability_df.dropna().reset_index(drop=True)
platemap_viability_df

,Treatment,Dose,Unit,patient_id,Drug,Concentration_uM,Viability_percentage
0,Staurosporine,10,nM,NF0014_T1,Staurosporine,10.0,5.515339
1,Digoxin,1,uM,NF0014_T1,Digoxin,1.0,22.499116
2,Digoxin,1,uM,NF0014_T1,Digoxin,1.0,22.499116
3,Onalespib,1,uM,NF0014_T1,Onalespib,1.0,32.645151
4,Staurosporine,10,nM,NF0014_T1,Staurosporine,10.0,5.515339
...,...,...,...,...,...,...,...
365,Staurosporine,10,nM,NF0055_T1,Staurosporine,10.0,4.276642
366,Selumetinib,1,uM,NF0055_T1,Selumetinib,1.0,108.513402
367,Selumetinib,10,uM,NF0055_T1,Selumetinib,10.0,108.783376
368,Selumetinib,10,uM,NF0055_T1,Selumetinib,10.0,108.783376


In [7]:
agg_profiles_path = pathlib.Path(
    f"{root_dir}/data/profiles_3D/all_patients/3.consensus_profiles/"
).resolve(strict=True)
agg_profiles_df_list = [x for x in agg_profiles_path.glob("*.parquet") if x.is_file()]

# Initialized ONCE, outside the per-profile loop, so results from every
# profile accumulate instead of being overwritten each iteration.
results = {
    "profile_type": [],
    "split_method": [],
    "shuffle_status": [],
    "results": [],
}

# Fixed seed so the "shuffled" permutation control is reproducible run-to-run.
rng = np.random.default_rng(0)

for consensus_path in agg_profiles_df_list:
    consensus_profile_name = consensus_path.stem
    print(f"Processing and training model for {consensus_profile_name}...")
    consensus_df = pd.read_parquet(consensus_path)

    viabilities_df = (
        pd.merge(
            consensus_df,
            platemap_viability_df,
            how="left",
            left_on=[
                "Metadata_Biology_PatientTumor",
                "Metadata_Experiment_Treatment",
                "Metadata_Experiment_Dose",
            ],
            right_on=["patient_id", "Treatment", "Dose"],
        )
        .drop(
            columns=[
                "Unit",
                "patient_id",
                "Drug",
                "Concentration_uM",
                "Treatment",
                "Dose",
            ]
        )
        .rename(
            columns={
                x: f"Metadata_{x}"
                for x in platemap_viability_df.columns
                if "Viability_percentage" not in x
                and x not in ["WellCol", "WellPosition", "Drug", "Concentration_uM"]
            }
        )
    )
    # drop NAN rows to avoid patients that do not have viability data
    viabilities_df = viabilities_df.dropna(subset=["Viability_percentage"]).reset_index(
        drop=True
    )
    # combine the two stratification columns into a single key
    viabilities_df["Metadata_Experiment_FullTreatment"] = (
        viabilities_df["Metadata_Experiment_Treatment"].astype(str)
        + "_"
        + viabilities_df["Metadata_Experiment_Dose"].astype(str)
    )
    viability_col = ["Viability_percentage"]
    metadata_cols = [
        col for col in viabilities_df.columns if col.startswith("Metadata_")
    ]
    feature_cols = [
        col
        for col in viabilities_df.columns
        if col not in metadata_cols and col not in viability_col
    ]
    viabilities_df[feature_cols] = viabilities_df[feature_cols].clip(
        lower=-1e10, upper=1e10
    )

    # down sample temporarily to speed up testing
    viabilities_df = viabilities_df.sample(n=24, random_state=0).reset_index(drop=True)

    for shuffle_status in ["not_shuffled", "shuffled"]:
        if shuffle_status == "shuffled":
            # permute the values in every column (fixed seed via rng, defined above)
            viabilities_df[feature_cols] = viabilities_df[feature_cols].apply(
                lambda col: rng.permutation(col.values)
            )
        for split_method in (
            "lopo",
            "loto",
            "random_split",
        ):
            if split_method == "lopo":
                group_col = PATIENT_COL
            elif split_method == "loto":
                group_col = TREATMENT_COL
            else:
                group_col = None  # not used for "random_split"

            if split_method == "random_split":
                fold_result = run_random_split(
                    viabilities_df,
                    feature_cols,
                    viability_col,
                    split_name=split_method,
                    test_size=RANDOM_SPLIT_TEST_SIZE,
                    random_state=RANDOM_SPLIT_SEED,
                    shuffle_status=shuffle_status,
                    profile_type=consensus_profile_name,
                )
            else:
                fold_result = run_group_cv(
                    viabilities_df,
                    feature_cols,
                    viability_col,
                    group_col=group_col,
                    split_name=split_method,
                    shuffle_status=shuffle_status,
                    profile_type=consensus_profile_name,
                )

            results["results"].append(fold_result)
            results["profile_type"].append(consensus_profile_name)
            results["split_method"].append(split_method)
            results["shuffle_status"].append(shuffle_status)

Processing and training model for nucleocentric_morphem_norm_sc_consensus_profiles...
Any inf in X: False
Any NaN in X: False
Max abs value in X: 4.414443435974015
Columns with inf values: []

=== lopo grouped CV on 'Metadata_Biology_PatientTumor' (6 folds) for not_shuffled ===
  Fold 0 (held out=NF0014_T1, n=21, train): R2=0.8400, RMSE=16.0634
  Fold 0 (held out=NF0014_T1, n=3, test): R2=0.6029, RMSE=19.0562
  Fold 1 (held out=NF0014_T2, n=21, train): R2=0.8727, RMSE=14.9343
  Fold 1 (held out=NF0014_T2, n=3, test): R2=-4.3773, RMSE=41.0576
  Fold 2 (held out=NF0018_T6, n=18, train): R2=0.8477, RMSE=17.1239
  Fold 2 (held out=NF0018_T6, n=6, test): R2=0.1693, RMSE=26.8559
  Fold 3 (held out=NF0035_T1, n=19, train): R2=0.7407, RMSE=19.3762
  Fold 3 (held out=NF0035_T1, n=5, test): R2=0.4428, RMSE=36.2688
  Fold 4 (held out=NF0037_T1, n=19, train): R2=0.8213, RMSE=17.6512
  Fold 4 (held out=NF0037_T1, n=5, test): R2=0.9426, RMSE=8.8470
  Fold 5 (held out=NF0055_T1, n=22, train): R2=0.87

/home/lippincm/Documents/NF1_organoid_profile_analysis/.venv/lib/python3.13/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


  Fold 4 (held out=Fimepinostat_1, n=23, train): R2=0.8657, RMSE=15.2939
  Fold 4 (held out=Fimepinostat_1, n=1, test): R2=nan, RMSE=58.1877


/home/lippincm/Documents/NF1_organoid_profile_analysis/.venv/lib/python3.13/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


  Fold 5 (held out=Linsitinib_1, n=23, train): R2=0.7522, RMSE=20.7816
  Fold 5 (held out=Linsitinib_1, n=1, test): R2=nan, RMSE=17.4728


...

  Fold 6 (held out=Mirdametinib_1, n=22, train): R2=0.8209, RMSE=17.9764
  Fold 6 (held out=Mirdametinib_1, n=2, test): R2=-8.0910, RMSE=19.9919
  Fold 7 (held out=Rapamycin_1, n=23, train): R2=0.6915, RMSE=22.7361
  Fold 7 (held out=Rapamycin_1, n=1, test): R2=nan, RMSE=28.4451


/home/lippincm/Documents/NF1_organoid_profile_analysis/.venv/lib/python3.13/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


  Fold 8 (held out=Selumetinib_1, n=23, train): R2=0.5744, RMSE=27.1014


/home/lippincm/Documents/NF1_organoid_profile_analysis/.venv/lib/python3.13/site-packages/sklearn/metrics/_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


  Fold 8 (held out=Selumetinib_1, n=1, test): R2=nan, RMSE=29.4356
  Fold 9 (held out=Selumetinib_10, n=20, train): R2=0.0000, RMSE=38.0361
  Fold 9 (held out=Selumetinib_10, n=4, test): R2=-3.0980, RMSE=56.5029
  Fold 10 (held out=Staurosporine_10, n=19, train): R2=0.3486, RMSE=18.0180
  Fold 10 (held out=Staurosporine_10, n=5, test): R2=-11673.1301, RMSE=86.5467

--- loto summary (across 11 folds, test set) ---
               R2          MSE        MAE        MAPE       RMSE
mean -1963.130078  1385.710572  27.232668  162.678245  28.615223
std   4757.049695  2243.088256  24.363453  464.959654  24.971335
Any inf in X: False
Any NaN in X: False
Max abs value in X: 1.658108115196228
Columns with inf values: []

=== random_split (70/30 train/test, n_train=16, n_test=8) for shuffled ===
  train (n=16): R2=0.8522, RMSE=16.2121
  test (n=8): R2=-0.0478, RMSE=38.3728

--- random_split summary (single split, test set) ---
            R2          MSE        MAE        MAPE       RMSE
mean -0.04

In [8]:
results

{'profile_type': ['nucleocentric_morphem_norm_sc_consensus_profiles',
  'nucleocentric_morphem_norm_sc_consensus_profiles',
  'nucleocentric_morphem_norm_sc_consensus_profiles',
  'nucleocentric_morphem_norm_sc_consensus_profiles',
  'nucleocentric_morphem_norm_sc_consensus_profiles',
  'nucleocentric_morphem_norm_sc_consensus_profiles',
  'sammed_organoid_norm_sc_consensus_profiles',
  'sammed_organoid_norm_sc_consensus_profiles',
  'sammed_organoid_norm_sc_consensus_profiles',
  'sammed_organoid_norm_sc_consensus_profiles',
  'sammed_organoid_norm_sc_consensus_profiles',
  'sammed_organoid_norm_sc_consensus_profiles',
  'sc_norm_sc_consensus_profiles',
  'sc_norm_sc_consensus_profiles',
  'sc_norm_sc_consensus_profiles',
  'sc_norm_sc_consensus_profiles',
  'sc_norm_sc_consensus_profiles',
  'sc_norm_sc_consensus_profiles',
  'organoid_norm_sc_consensus_profiles',
  'organoid_norm_sc_consensus_profiles',
  'organoid_norm_sc_consensus_profiles',
  'organoid_norm_sc_consensus_profiles'

In [9]:
pd.read_parquet(
    pathlib.Path(
        "../model_results/lopo_predicted_viabilities__not_shuffled__sammed_nucleocentric_norm_sc_consensus_profiles.parquet"
    )
)

,Metadata_Biology_PatientTumor,Metadata_Experiment_Treatment,Metadata_Experiment_Dose,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Unit,Nucleocentric_AGP_SAMMed3D_Feature0,Nucleocentric_AGP_SAMMed3D_Feature1,Nucleocentric_AGP_SAMMed3D_Feature10,...,Nucleocentric_Mito_SAMMed3D_Feature96,Nucleocentric_Mito_SAMMed3D_Feature97,Nucleocentric_Mito_SAMMed3D_Feature98,Nucleocentric_Mito_SAMMed3D_Feature99,Viability_percentage,Metadata_Experiment_FullTreatment,Actual_Viability,Predicted_Viability,Metadata_fold,Metadata_held_out_group
0,NF0014_T1,Staurosporine,10,Small Molecule,Apoptosis induction,Experimental,nM,-0.449589,-1.113767,0.714894,...,-0.834729,0.458213,-0.073315,0.375830,5.515339,Staurosporine_10,5.515339,69.596322,0,NF0014_T1
1,NF0014_T1,Linsitinib,1,Small Molecule,IGF-1R inhibitor,Investigational,uM,-0.050384,-0.244386,0.241700,...,0.080646,-0.064007,-0.275601,-0.053719,79.531230,Linsitinib_1,79.531230,97.499435,0,NF0014_T1
2,NF0014_T1,Copanlisib,1,Small Molecule,PI3K inhibitor,Kinase Inhibitor,uM,-0.291448,-0.651557,0.344767,...,-0.007808,0.308240,-0.389865,0.000395,45.043833,Copanlisib_1,45.043833,61.067996,0,NF0014_T1
3,NF0014_T2,Mirdametinib,1,Small Molecule,MEK1/2 inhibitor,Kinase Inhibitor,uM,-0.094685,-0.382230,0.341613,...,-0.276874,0.700693,0.092751,0.330359,80.777050,Mirdametinib_1,80.777050,28.128755,1,NF0014_T2
4,NF0014_T2,Selumetinib,10,Small Molecule,MEK1/2 inhibitor,Kinase Inhibitor,uM,-0.017732,-0.005597,-0.406061,...,-0.006818,0.283831,0.073561,0.095566,124.056923,Selumetinib_10,124.056923,66.138986,1,NF0014_T2
5,NF0014_T2,DMSO,1,Control,Control,Control,%,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,100.000000,DMSO_1,100.000000,93.251278,1,NF0014_T2
6,NF0018_T6,Copanlisib,1,Small Molecule,PI3K inhibitor,Kinase Inhibitor,uM,0.147950,-0.199917,0.198630,...,0.009871,-0.002281,-0.126648,0.248119,76.290346,Copanlisib_1,76.290346,50.326766,2,NF0018_T6
7,NF0018_T6,Fimepinostat,1,Small Molecule,PI3K and HDAC inhibitor,Investigational,uM,-0.025592,-0.546546,0.511470,...,-0.326295,0.421039,-0.180315,-0.165073,70.318499,Fimepinostat_1,70.318499,22.901508,2,NF0018_T6
8,NF0018_T6,Selumetinib,10,Small Molecule,MEK1/2 inhibitor,Kinase Inhibitor,uM,-0.230507,-0.254793,0.398020,...,0.100643,0.180156,-0.087472,-0.136363,91.120432,Selumetinib_10,91.120432,77.834896,2,NF0018_T6
9,NF0018_T6,Binimetinib,1,Small Molecule,MEK1/2 inhibitor,Kinase Inhibitor,uM,0.017104,-0.178551,0.147404,...,0.204087,0.012286,0.008969,-0.191729,84.800228,Binimetinib_1,84.800228,76.504424,2,NF0018_T6
